# Notebook Analyse Go/NoGo

Ce carnet guide l'analyse des potentiels évoqués (ERP) pour le paradigme Go/NoGo.
Il reprend la structure "Type 1 / Type 2 / Type 3" introduite dans `01_preprocessing_notebook.ipynb` :
- **Type 1 — prêt à exécuter** : cellules complètes, sans modification nécessaire.
- **Type 2 — à personnaliser** : blocs contenant des paramètres à ajuster (balises `A_COMPLETER`).
- **Type 3 — exploration libre** : propositions d'analyses supplémentaires, prêtes à modifier.

## Pour bien démarrer
- Téléchargez / montez le dossier `tasks/gonogo/bids` au même niveau que ce notebook, ou adaptez `root_bids`.
- Assurez-vous d'avoir exécuté le pipeline de prétraitement pour générer les fichiers `*_processed.fif`.
- Activez l'environnement Python du cours et installez les dépendances (`pip install -r requirements.txt`).

## Objectifs pédagogiques
1. Charger un enregistrement Go/NoGo déjà prétraité.
2. Extraire les événements, découper en epochs et calculer les ERPs Go vs NoGo.
3. Explorer des métriques temporelles (amplitude moyenne, latence de pics) et comparer plusieurs sujets ou données de groupe.


## 0. Préparation et configuration

Nous commençons par configurer l'environnement d'analyse (imports, chemins, sélection du sujet).


### Bloc Type 1 — Imports et options globales

Initialise les bibliothèques nécessaires, masque certains avertissements MNE et fixe le style de figures Matplotlib.


In [ ]:
# -----------------------------------------------------------------------------
# Imports principaux et configuration globale
# Chaque instruction est commentée pour rappeler son rôle.
# -----------------------------------------------------------------------------
import warnings  # contrôle de l'affichage des avertissements Python
from pathlib import Path  # manipulation portable des chemins
import csv  # lecture de fichiers tabulés (.tsv)
import json  # sauvegarde des résultats intermédiaires

import matplotlib.pyplot as plt  # production des figures
import mne  # bibliothèque cœur pour les analyses EEG/MEG
import numpy as np  # opérations numériques vectorisées
import mne_bids  # outils pour gérer la structure BIDS avec MNE
from mne_bids import BIDSPath  # construction de chemins compatibles BIDS

warnings.filterwarnings('ignore', category=RuntimeWarning)  # masque certains avertissements MNE
mne.set_log_level('INFO')  # verbosité modérée pour suivre les étapes clés
plt.rcParams['figure.figsize'] = (10, 5)  # taille par défaut des figures Matplotlib

print('Versions utilisées:')  # journalise les versions pour la reproductibilité
print(' - mne      ', mne.__version__)  # version de MNE
print(' - numpy    ', np.__version__)  # version de NumPy
print(' - mne_bids ', mne_bids.__version__)  # version de mne-bids


### Bloc Type 1 — Définir le dossier BIDS et les dérivés

Localise le dossier BIDS (données brutes) ainsi que les dérivés produits par le pipeline :
- `derivatives/preproc` pour les fichiers prétraités (sortie du notebook 01),
- `derivatives/gonogo-erp` pour les ERPs de groupe fournis,
- `derivatives/gonogo-analysis` pour les résultats sauvegardés par ce notebook.


In [ ]:
# -----------------------------------------------------------------------------
# Localisation des données BIDS et des dérivés nécessaires
# -----------------------------------------------------------------------------
root_bids = Path('../../data/gonogo/bids')  # chemin relatif recommandé

print('Chemins vérifiés:')  # confirmation dans la console
print(' - BIDS root            :', root_bids.resolve())  # chemin absolu utilisé

deriv_preproc = root_bids / 'derivatives' / 'preproc'  # mêmes dérivés que le notebook de prétraitement
deriv_preproc.mkdir(parents=True, exist_ok=True)

deriv_erp = root_bids / 'derivatives' / 'gonogo-erp'  # ERP de groupe fournis

deriv_analysis = root_bids / 'derivatives' / 'gonogo-analysis'  # résultats spécifiques à ce notebook
deriv_analysis.mkdir(parents=True, exist_ok=True)
print(' - Dérivés (preproc)    :', deriv_preproc.resolve())
print(' - Dérivés (analysis)   :', deriv_analysis.resolve())


### Bloc Type 1 — Lister les participants disponibles

Lit `participants.tsv` (convention BIDS) pour récupérer les identifiants `sub-XX` disponibles. 
En Type 2 ci-dessous, nous choisirons l'un de ces sujets.


In [ ]:
# -----------------------------------------------------------------------------
# Lecture de participants.tsv afin d'obtenir la liste des sujets présents
# -----------------------------------------------------------------------------
participants_tsv = root_bids / 'participants.tsv'  # chemin vers le fichier BIDS
subjects = []  # contiendra les identifiants sans le préfixe 'sub-'

with participants_tsv.open('r', encoding='utf-8') as f:  # ouverture du fichier en lecture
    reader = csv.reader(f, delimiter='	')  # lecture tabulée
    header = next(reader, None)  # saute l'entête (participant_id, ...)
    for row in reader:  # boucle sur chaque ligne restante
        if not row:  # ignore les lignes vides
            continue
        participant_id = row[0]  # première colonne = identifiant sujet
        if participant_id.startswith('sub-'):  # vérifie le format BIDS
            subjects.append(participant_id.replace('sub-', ''))  # stocke l'identifiant sans préfixe

print(f"Participants détectés ({len(subjects)}): {subjects}")  # affiche la liste obtenue


### Bloc Type 2 — Sélectionner un participant et une session

Modifiez les variables ci-dessous pour cibler un autre enregistrement. 
Les valeurs proposées correspondent à un exemple valide, mais n'hésitez pas à tester différents sujets.


In [ ]:
# -----------------------------------------------------------------------------
# Choix du sujet/session/run à analyser (modifiable)
# -----------------------------------------------------------------------------
subject = '01'  # <--- remplacez par ex. '05' pour explorer un autre participant
session = '001'  # la plupart des sujets possèdent uniquement cette session
run = '01'  # un seul run Go/NoGo disponible

print(f'Sujet en cours: sub-{subject}, session {session}, run {run}')  # trace la sélection


## 1. Charger un enregistrement prétraité

Nous chargeons le fichier `*_processed.fif` issu du pipeline précédent, appliquons un montage standard et vérifions les métadonnées clés.


### Bloc Type 1 — Fonction utilitaire de chargement

`load_processed_raw` centralise la construction du chemin de fichier, le chargement `Raw` MNE et l'application d'un montage 10-20 international.


In [ ]:
# -----------------------------------------------------------------------------
# Fonction utilitaire : chargement d'un fichier prétraité pour un sujet donné
# -----------------------------------------------------------------------------
def load_processed_raw(subject: str, session: str = '001', run: str = '01') -> mne.io.BaseRaw:
    # Construit le chemin BIDS du fichier *_processed.fif dans derivatives/preproc
    processed_bids = BIDSPath(
        root=deriv_preproc,
        subject=subject,
        session=session,
        task='gonogo',
        run=run,
        datatype='eeg',
        processing='clean',
        suffix='processed',
        extension='.fif'
    )
    fname = processed_bids.fpath
    if not fname.exists():  # garde-fou si le fichier manque
        raise FileNotFoundError(f'Fichier introuvable: {fname}')  # message explicite
    raw_obj = mne.io.read_raw_fif(fname, preload=True)  # charge en mémoire pour un accès rapide
    raw_obj.set_montage('standard_1020', match_case=False, on_missing='warn')  # assure la co-registration EEG
    return raw_obj  # renvoie l'objet Raw prêt à l'emploi

raw = load_processed_raw(subject, session=session, run=run)  # chargement effectif
print(raw)  # résumé de l'objet Raw


### Bloc Type 1 — Vérifier les annotations et métadonnées

Affiche la fréquence d'échantillonnage, la liste des canaux EEG, ainsi que les annotations importées (événements détectés et autres marquages).


In [ ]:
# -----------------------------------------------------------------------------
# Inspection rapide des métadonnées pour valider le chargement
# -----------------------------------------------------------------------------
print('Fréquence échantillonnage :', raw.info['sfreq'], 'Hz')  # vérifie la fréquence
print('Canaux EEG                :', [ch for ch in raw.ch_names if ch not in ('Trigger', 'Event')])  # liste EEG
print('Annotations disponibles   :', sorted(set(raw.annotations.description)))  # types d'événements
print('Nombre total annotations  :', len(raw.annotations))  # quantité d'annotations


### Bloc Type 2 — Visualisation rapide du signal brut (optionnel)

Décommentez la ligne suivante pour afficher quelques secondes de signal. 
Ajustez `n_channels`, `scalings` ou la fenêtre temporelle selon vos besoins.


In [ ]:
# raw.copy().crop(tmax=5).plot(n_channels=12, scalings='auto')


## 2. Détection des événements et création des epochs

Nous mappons les annotations BIDS vers des étiquettes lisibles (`go/onset`, `nogo/onset`, ...), puis nous découpons les données en epochs alignés sur ces événements.


### Bloc Type 1 — Extraire les événements Go/NoGo depuis les annotations

La table `annotation_map` relie les identifiants bruts (`Stimulus/S  4`, `Stimulus/S  5`, ...) à des labels plus explicites.


In [ ]:
# -----------------------------------------------------------------------------
# Extraction des événements et renommage en labels explicites
# -----------------------------------------------------------------------------
events, event_id = mne.events_from_annotations(raw)  # convertit les annotations en matrice d'événements
annotation_map = {
    'Stimulus/S  4': 'go/onset',
    'Stimulus/S  5': 'nogo/onset',
    'Stimulus/S  6': 'go/correct',
    'Stimulus/S  8': 'nogo/correct',
}

selected_event_id = {}
for original, label in annotation_map.items():  # parcours des correspondances
    if original in event_id:  # ne garde que les codes présents pour ce sujet
        selected_event_id[label] = event_id[original]  # stocke l'étiquette lisible

if not selected_event_id:  # garde-fou si aucune correspondance
    raise RuntimeError('Aucun événement Go/NoGo détecté — vérifiez annotation_map.')

print('Étiquettes retenues:')  # journalise la sélection
for label, code in selected_event_id.items():  # boucle affichage
    n_trials = int((events[:, 2] == code).sum())  # compte les essais correspondant au code
    print(f' - {label:13s} → code {code:2d}, essais = {n_trials}')  # sortie formatée


### Bloc Type 2 — Ajuster la correspondance codes ↔ labels (optionnel)

Si votre dataset inclut d'autres événements, complétez `annotation_map_custom` ci-dessous puis réexécutez la cellule précédente.


In [ ]:
# Exemple : personnaliser la table de correspondance événements ↔ labels
annotation_map_custom = {
    # 'Stimulus/S  9': 'autre_condition',  # décommentez / ajoutez vos propres mappings
}

annotation_map.update(annotation_map_custom)  # fusion avec le dictionnaire de base
print('Annotation map mise à jour:', annotation_map)  # vérification console


### Bloc Type 1 — Paramètres d'epoching

Définit la fenêtre temporelle autour des événements et calcule les epochs EEG associés.


In [ ]:
# -----------------------------------------------------------------------------
# Paramètres d'epoching et construction des epochs MNE
# -----------------------------------------------------------------------------
tmin, tmax = -0.2, 0.8  # fenêtre autour de l'événement (en secondes)
baseline = (-0.2, 0.0)  # correction de ligne de base (200 ms pré-stimulus)

epochs = mne.Epochs(
    raw,
    events,
    event_id=selected_event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=baseline,
    picks='eeg',
    preload=True,
    detrend=None,
)  # construit l'objet Epochs prêt à l'emploi
print(epochs)  # résumé (taille, n_epochs, etc.)
trial_counts = {cond: len(epochs[cond]) for cond in epochs.event_id}  # nombre d'essais par condition
print('Essais conservés par condition :', trial_counts)  # affichage console


### Bloc Type 2 — Visualiser quelques epochs (optionnel)

Décommentez le code ci-dessous pour afficher un extrait d'epochs et inspecter la qualité des essais.


In [ ]:
# epochs.copy().plot(n_epochs=10, n_channels=10, scalings='auto')  # décommentez pour inspecter visuellement


### Bloc Type 1 — Calculer les ERPs Go vs NoGo

Calcule la moyenne des epochs pour chaque condition et produit un tracé temporel.


In [ ]:
# -----------------------------------------------------------------------------
# Calcul des ERPs (Evoked) pour chaque condition retenue
# -----------------------------------------------------------------------------
evokeds = {label: epochs[label].average() for label in epochs.event_id.keys()}  # moyenne par condition
for label, evk in evokeds.items():  # boucle sur chaque ERP
    print(f"Evoked {label}: n_epochs={len(epochs[label])}, n_channels={len(evk.ch_names)}")  # résumé
    evk.plot(spatial_colors=True, time_unit='s', titles=f'ERP — {label}')  # tracé interactif


### Bloc Type 2 — Topographies temporelles


In [ ]:
# -----------------------------------------------------------------------------
# Topographies moyennes sur une fenêtre d'intérêt (200-450 ms)
# -----------------------------------------------------------------------------
if 'nogo/onset' in evokeds:  # trace NoGo si disponible
    evokeds['nogo/onset'].plot_topomap(
        times=np.linspace(0.2, 0.45, 5),  # instants d'intérêt
        ch_type='eeg',  # type de capteur
        time_unit='s',  # secondes
        colorbar=True  # ajoute une échelle
    )
if 'go/onset' in evokeds:  # trace Go si disponible
    evokeds['go/onset'].plot_topomap(
        times=np.linspace(0.2, 0.45, 5),  # mêmes instants pour comparer
        ch_type='eeg',
        time_unit='s',
        colorbar=True
    )


## 3. Comparer amplitudes et latences (temps, pas de fréquence)


### Bloc Type 1 — Fonctions d'extraction de métriques temporelles

- `mean_amplitude_microvolt` calcule la moyenne d'amplitude (µV) sur une fenêtre temporelle.
- `peak_latency` renvoie la latence et l'amplitude du pic positif (ou négatif) demandé.


In [ ]:
# -----------------------------------------------------------------------------
# Fonctions utilitaires : amplitude moyenne et latence de pic
# -----------------------------------------------------------------------------
def mean_amplitude_microvolt(evoked: mne.Evoked, picks, tmin: float, tmax: float) -> dict:
    # Retourne la moyenne en microvolts (µV) pour les canaux sélectionnés.
    picks = picks if isinstance(picks, (list, tuple)) else [picks]
    evk = evoked.copy().pick(picks)
    start, stop = evk.time_as_index([tmin, tmax])
    if stop <= start:
        raise ValueError('Fenêtre temporelle invalide pour mean_amplitude_microvolt.')
    segment = evk.data[:, start:stop]
    return {
        ch: float(segment[i].mean() * 1e6)  # moyenne µV par canal
        for i, ch in enumerate(evk.ch_names)
    }

def peak_latency(evoked: mne.Evoked, pick: str, tmin: float, tmax: float, mode: str = 'pos'):
    # Retourne (temps en secondes, amplitude en µV) du pic demandé.
    evk = evoked.copy().pick(pick)
    ch_name, time_s = evk.get_peak(tmin=tmin, tmax=tmax, mode=mode)
    time_idx = evk.time_as_index(time_s)
    amp = evk.data[0, time_idx]
    return float(time_s), float(amp * 1e6)


### Bloc Type 2 — Appliquer sur Cz (modifiable)

Fenêtre cible : 250–400 ms (typiquement composante P3). Modifiez `channel_of_interest` ou la fenêtre pour explorer d'autres effets.


In [ ]:
# -----------------------------------------------------------------------------
# Calcul de métriques (moyenne et pic) sur une fenêtre d'intérêt
# -----------------------------------------------------------------------------
time_window = (0.25, 0.40)  # fenêtre en secondes (typiquement P3)
channel_of_interest = 'Cz'  # canal EEG étudié
metrics = {}  # contiendra les résultats par condition
for label in ('go/onset', 'nogo/onset'):  # boucle sur les conditions principales
    if label in evokeds:  # continue si l'ERP est disponible
        amps = mean_amplitude_microvolt(evokeds[label], channel_of_interest, *time_window)  # amplitude moyenne
        peak_t, peak_amp = peak_latency(evokeds[label], channel_of_interest, *time_window, mode='pos')  # pic positif
        metrics[label] = {
            'mean_amp_uV': amps[channel_of_interest],
            'peak_time_s': peak_t,
            'peak_amp_uV': peak_amp,
        }

subject_dir = deriv_analysis / f'sub-{subject}'
subject_dir.mkdir(parents=True, exist_ok=True)
session_label = session if session is not None else 'NA'
metrics_path = subject_dir / f'sub-{subject}_ses-{session_label}_task-gonogo_metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print('Metrics sauvegardees dans', metrics_path)

metrics  # affiche le dictionnaire de résultats


### Bloc Type 3 — Différence NoGo - Go

Calcule et visualise la différence NoGo–Go (vague de différence). 
Ajoutez d'autres contrastes selon vos besoins.


In [ ]:
# -----------------------------------------------------------------------------
# Vague de différence NoGo - Go et amplitude moyenne sur la fenêtre choisie
# -----------------------------------------------------------------------------
if {'go/onset', 'nogo/onset'}.issubset(evokeds):  # vérifie la disponibilité des deux ERPs
    difference = mne.combine_evoked(
        [evokeds['nogo/onset'], evokeds['go/onset']],  # liste des ERPs à combiner
        weights=[1, -1]  # NoGo - Go
    )
    difference.plot(
        spatial_colors=True,
        time_unit='s',
        titles='Difference wave — NoGo minus Go'
    )
    diff_metrics = mean_amplitude_microvolt(difference, channel_of_interest, *time_window)  # amplitude moyenne
    diff_path = subject_dir / f'sub-{subject}_ses-{session_label}_task-gonogo_nogo-minus-go-ave.fif'
    difference.save(diff_path, overwrite=True)
    print('Difference sauvegardee dans', diff_path)
    print(f"Amplitude moyenne Δ(NoGo-Go) sur {channel_of_interest}: {diff_metrics[channel_of_interest]:.3f} µV")


### Bloc Type 3 — Explorer d'autres métriques (optionnel)

Utilisez ce canevas pour tester d'autres canaux ou fenêtres temporelles (ex. N2, P1). Ajoutez vos propres analyses.


In [ ]:
custom_windows = {
    # 'N2_Cz': {'channel': 'Cz', 'window': (0.18, 0.26)},  # exemple à personnaliser
}

for label in ('go/onset', 'nogo/onset'):  # boucle sur les conditions disponibles
    if label not in evokeds:  # ignore si l'ERP est absent
        continue
    for metric_name, params in custom_windows.items():  # boucle sur les métriques à calculer
        ch = params['channel']  # canal cible
        tmin_custom, tmax_custom = params['window']  # fenêtre temporelle
        amp = mean_amplitude_microvolt(evokeds[label], ch, tmin_custom, tmax_custom)[ch]  # amplitude moyenne
        print(f"{label} — {metric_name}: {amp:.3f} µV")  # affiche le résultat


## 4. Extension multi-participants

Automatise l'extraction de la composante P3 pour tous les sujets détectés. 
Utile pour créer rapidement un tableau récapitulatif ou préparer une analyse statistique groupée.


### Bloc Type 3 — Boucler sur plusieurs sujets

Exécute automatiquement les étapes précédentes pour chaque participant et affiche l'amplitude moyenne Go/NoGo sur Cz.


In [ ]:
# -----------------------------------------------------------------------------
# Passage sur tous les sujets : extraction de la moyenne P3 sur Cz
# -----------------------------------------------------------------------------
summary = []  # stockera les résultats par sujet
for sub in subjects:  # boucle sur chaque participant détecté
    try:
        raw_sub = load_processed_raw(sub, session=session, run=run)  # charge le fichier prétraité
        events_sub, event_id_sub = mne.events_from_annotations(raw_sub)  # récupère les événements
        selected_sub = {label: event_id_sub[src] for src, label in annotation_map.items() if src in event_id_sub}  # filtre
        if {'go/onset', 'nogo/onset'}.issubset(selected_sub.keys()):  # continue si les deux conditions sont présentes
            epochs_sub = mne.Epochs(
                raw_sub,
                events_sub,
                event_id=selected_sub,
                tmin=tmin,
                tmax=tmax,
                baseline=baseline,
                picks='eeg',
                preload=True,
                detrend=None,
            )
            go_ev = epochs_sub['go/onset'].average()  # ERP Go
            nogo_ev = epochs_sub['nogo/onset'].average()  # ERP NoGo
            go_amp = mean_amplitude_microvolt(go_ev, channel_of_interest, *time_window)[channel_of_interest]  # amplitude Go
            nogo_amp = mean_amplitude_microvolt(nogo_ev, channel_of_interest, *time_window)[channel_of_interest]  # amplitude NoGo
            summary.append((sub, go_amp, nogo_amp, nogo_amp - go_amp))  # enregistre les valeurs
    except FileNotFoundError:
        print(f'sub-{sub}: fichier prétraité manquant, ignoré.')  # message en cas d'absence de fichier

print('Résultats (µV) — fenêtre 250-400 ms sur', channel_of_interest)  # titre du résumé
for sub, go_amp, nogo_amp, delta in summary:  # affiche chaque ligne
    print(f"sub-{sub}: Go={go_amp:6.3f}  NoGo={nogo_amp:6.3f}  Δ(NoGo-Go)={delta:6.3f}")

if summary:
    group_csv = deriv_analysis / f'group_metrics_ses-{session_label}.csv'
    with group_csv.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['subject', 'go_mean_uV', 'nogo_mean_uV', 'delta_uV'])
        for sub, go_amp, nogo_amp, delta in summary:
            writer.writerow([sub, f"{go_amp:.6f}", f"{nogo_amp:.6f}", f"{delta:.6f}"])
    print('Tableau groupe sauvegarde :', group_csv)
else:
    print('Aucun sujet disponible pour le résumé groupe.')


### Bloc Type 1 — Charger les moyennes disponibles dans le dossier `gonogo-erp`

Charge les fichiers de groupe (`go`, `nogo`, `nogo-minus-go`) pour comparer vos résultats individuels aux ERP fournis avec le dataset.


## 5. Comparaison avec les ERP pré-calculés (@gonogo_erp)

Permet de confronter vos résultats individuels aux ERP fournis dans les dérivés `gonogo-erp`.


In [ ]:
# -----------------------------------------------------------------------------
# Chargement et visualisation des ERP de groupe (si disponibles)
# -----------------------------------------------------------------------------
group_go_path = deriv_erp / 'group' / 'erp' / 'group_task-gonogo_cond-go_evoked-ave.fif'  # chemin ERP Go
group_nogo_path = deriv_erp / 'group' / 'erp' / 'group_task-gonogo_cond-nogo_evoked-ave.fif'  # chemin ERP NoGo
group_diff_path = deriv_erp / 'group' / 'erp' / 'group_task-gonogo_cond-nogo-minus-go_evoked-ave.fif'  # différence NoGo-Go

if group_go_path.exists() and group_nogo_path.exists():  # vérifie la présence des fichiers de groupe
    group_go = mne.read_evokeds(group_go_path, verbose='ERROR')[0]  # charge ERP Go
    group_nogo = mne.read_evokeds(group_nogo_path, verbose='ERROR')[0]  # charge ERP NoGo
    mne.viz.plot_compare_evokeds(
        {'Go (groupe)': group_go, 'NoGo (groupe)': group_nogo},  # dictionnaire conditions
        picks='Cz',  # canal d'intérêt
        combine='mean',  # moyenne sur le canal
        colors={'Go (groupe)': 'tab:green', 'NoGo (groupe)': 'tab:red'},  # couleurs personnalisées
        truncate_yaxis='auto',  # adapte l'échelle
        show=True,  # affiche la figure immédiatement
        title='ERP de groupe — Cz'  # titre de la figure
    )
if group_diff_path.exists():  # si la différence est disponible
    group_diff = mne.read_evokeds(group_diff_path, verbose='ERROR')[0]  # charge la vague de différence
    group_diff.plot(
        spatial_colors=True,
        time_unit='s',
        titles='Difference wave groupe — NoGo minus Go'
    )
